<a href="https://colab.research.google.com/github/ozgurbaybas/newRepo/blob/main/vLLM_Serve_AWQ_and_SqueezeLLM_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [vLLM: Serve Fast Mistral 7B and Llama 2 Models from Your Computer](https://newsletter.kaitchup.com/p/vllm-serve-fast-mistral-7b-and-llama)*

This notebook shows how to run and serve LLMs, here, Mistral 7B and Llama 2 7B, quantized with AWQ and SqueezeLLM.

It requires a GPU with at least 16 GB of VRAM and CUDA 12.1.

First, we need to install vLLM

In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 39.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.2/307.2 kB 39.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 MB 25.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 99.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 kB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 65.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 66.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 93.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━

# Offline Inference

For offline inference, i.e., without starting a server, run the following cell.

Example with AWQ:

In [ ]:
import time
from vllm import LLM, SamplingParams

prompts = [
    "The best recipe for a chicken curry is",
    "2 + 2 =",
    "The color of the blue car is",
    "This is the Pythorch implementation of a violin:",
]
sampling_params = SamplingParams(temperature=0.7, top_p=0.9)

loading_start = time.time()
llm = LLM(model="kaitchup/Mistral-7B-awq-4bit", quantization="awq")
print("--- Loading time: %s seconds ---" % (time.time() - loading_start))

generation_time = time.time()
outputs = llm.generate(prompts, sampling_params)
print("--- Generation time: %s seconds ---" % (time.time() - generation_time))

for output in outputs:
    generated_text = output.outputs[0].text
    print(generated_text)
    print('------')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

WARNING 02-15 04:00:26 config.py:177] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 02-15 04:00:26 llm_engine.py:72] Initializing an LLM engine with config: model='kaitchup/Mistral-7B-awq-4bit', tokenizer='kaitchup/Mistral-7B-awq-4bit', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, seed=0)


tokenizer_config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

INFO 02-15 04:00:34 weight_utils.py:164] Using model weights format ['*.safetensors']


model.safetensors:   0%|          | 0.00/4.15G [00:00<?, ?B/s]

INFO 02-15 04:00:50 llm_engine.py:322] # GPU blocks: 13825, # CPU blocks: 2048
INFO 02-15 04:00:52 model_runner.py:632] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 02-15 04:00:52 model_runner.py:636] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 02-15 04:00:59 model_runner.py:698] Graph capturing finished in 7 secs.
--- Loading time: 35.187790393829346 seconds ---


Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 17.34it/s]

--- Generation time: 0.23667311668395996 seconds ---
 the one you like best.

The best way to cook a chicken cur
------
 4

This is the first equation I ever learned.

This
------
 called Tangy Orange.

I have a blue car and I am
------


The idea of the violin is to play a continuous note while a
------


Example with SqueezeLLM:

In [ ]:
import time
from vllm import LLM, SamplingParams

prompts = [
    "The best recipe for a chicken curry is",
    "2 + 2 =",
    "The color of the blue car is",
    "This is the Pythorch implementation of a violin:",
]
sampling_params = SamplingParams(temperature=0.7, top_p=0.9)

loading_start = time.time()
llm = LLM(model="squeeze-ai-lab/sq-llama-2-7b-w4-s0", quantization="squeezellm")
print("--- Loading time: %s seconds ---" % (time.time() - loading_start))

generation_time = time.time()
outputs = llm.generate(prompts, sampling_params)
print("--- Generation time: %s seconds ---" % (time.time() - generation_time))

for output in outputs:
    generated_text = output.outputs[0].text
    print(generated_text)
    print('------')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/554 [00:00<?, ?B/s]

WARNING 02-15 04:05:38 config.py:177] squeezellm quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 02-15 04:05:38 llm_engine.py:72] Initializing an LLM engine with config: model='squeeze-ai-lab/sq-llama-2-7b-w4-s0', tokenizer='squeeze-ai-lab/sq-llama-2-7b-w4-s0', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=squeezellm, enforce_eager=False, kv_cache_dtype=auto, seed=0)


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

INFO 02-15 04:05:45 weight_utils.py:164] Using model weights format ['*.pt']


sq-llama-2-7b-w4-s0.pt:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

INFO 02-15 04:06:19 llm_engine.py:322] # GPU blocks: 3956, # CPU blocks: 512
INFO 02-15 04:06:22 model_runner.py:632] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 02-15 04:06:22 model_runner.py:636] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 02-15 04:06:41 model_runner.py:698] Graph capturing finished in 19 secs.
--- Loading time: 64.45989680290222 seconds ---


Processed prompts: 100%|██████████| 4/4 [00:00<00:00,  8.53it/s]

--- Generation time: 0.4749417304992676 seconds ---
 the one that tastes the best and gives you a sense of satisfaction.
------
 5, 2 + 2 = 10, 2 +
------
 sky blue. The color of the black car is black. The color of the
------
 a two-dimensional function that takes an input and outputs a probability distribution. The
------


# Serving LLMs

We can run vLLM in the background as follows:

In [ ]:
!nohup python -m vllm.entrypoints.openai.api_server --model kaitchup/Mistral-7B-awq-4bit --quantization awq &

nohup: appending output to 'nohup.out'


In [ ]:
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.


vLLM uses the OpenAI API's protocol to query the server. It works the same as if you were querying GPTs but instead we set a base_url and a fake API key.

*This doesn't communicate with OpenAI.*

In [ ]:
from openai import OpenAI

# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

prompts = [
    "The best recipe for a chicken curry is",
    "2 + 2 =",
    "The color of the blue car is",
    "This is the Pythorch implementation of a violin:",
]

completion = client.completions.create(model="kaitchup/Mistral-7B-awq-4bit",
                                      prompt=prompts)
print("Completion result:", completion)


Completion result: Completion(id='cmpl-e7eebd8dd8dd4ea896b2f865b0920aa8', choices=[CompletionChoice(finish_reason='length', index=0, logprobs=None, text=' one title of this great book of recipes. Next, there is the comprehensive section'), CompletionChoice(finish_reason='length', index=1, logprobs=None, text=' 4. 3+3=6. 6+6=1'), CompletionChoice(finish_reason='length', index=2, logprobs=None, text=' "Economy Blue Metallic" available since 2014'), CompletionChoice(finish_reason='length', index=3, logprobs=None, text='\n\n- [जोई, 2015](https://')], created=2267, model='kaitchup/Mistral-7B-awq-4bit', object='text_completion', system_fingerprint=None, usage=CompletionUsage(completion_tokens=64, prompt_tokens=39, total_tokens=103))
